# 1. Ingest

Pulls the finished feature mart straight out of Snowflake -- `fct_demand_features`, built by dbt (`dbt/gridcast/models/marts/fct_demand_features.sql`). All lag / rolling / calendar feature engineering already happened there, in SQL. This notebook's only job is landing that table as a local CSV so the rest of the pipeline (clean -> featurize -> train) can run with `dvc repro` without needing a warehouse connection every time.

**Where feature engineering actually lives:** dbt (SQL), not here. What notebook 3 (`03_features`) does with this data is a different job -- turning point-in-time features into a horizon-labeled training set -- explained there.

In [14]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [15]:
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path)

print("PROJECT_ROOT:", PROJECT_ROOT)
print(".env exists:", env_path.exists())
print("SNOWFLAKE_ACCOUNT:", os.getenv("SNOWFLAKE_ACCOUNT"))

PROJECT_ROOT: /Users/devashish/Desktop/Projects/gridcast
.env exists: True
SNOWFLAKE_ACCOUNT: AQWYEDW-ZY11947


In [16]:
from src.warehouse.snowflake_client import read_sql, schema_for

In [17]:
query = f"""
SELECT *
FROM {schema_for('marts')}.FCT_DEMAND_FEATURES
"""

df = read_sql(query)
df.columns = df.columns.str.lower()

df.head()

,demand_hour_utc,respondent,demand_mwh,forecast_mwh,demand_lag_1h,demand_lag_24h,demand_lag_168h,demand_rolling_mean_24h,demand_rolling_std_24h,demand_rolling_mean_168h,demand_rolling_std_168h,hour_of_day,day_of_week,month,is_weekend,is_holiday
0,2026-03-13 04:00:00,CISO,27966.0,28303.0,29107.0,26361.0,25992.0,24912.708333,2687.130059,23312.041667,2121.830653,4,5,3,False,False
1,2026-03-13 07:00:00,CISO,23662.0,24089.0,25250.0,22712.0,23604.0,25069.708333,2767.354591,23335.976190,2157.023948,7,5,3,False,False
2,2026-03-13 08:00:00,CISO,23045.0,22855.0,23662.0,22154.0,22230.0,25109.291667,2738.811492,23336.321429,2157.071744,8,5,3,False,False
3,2026-03-13 09:00:00,CISO,21998.0,21831.0,23045.0,21167.0,21943.0,25146.416667,2702.812592,23341.172619,2155.484632,9,5,3,False,False
4,2026-03-13 10:00:00,CISO,21243.0,21027.0,21998.0,20675.0,21142.0,25181.041667,2654.508085,23341.500000,2155.275169,10,5,3,False,False


In [18]:
print(
    f"{len(df):,} rows, "
    f"{df['respondent'].nunique()} regions, "
    f"{df['demand_hour_utc'].min()} -> {df['demand_hour_utc'].max()}"
)

df.columns.tolist()

131,402 rows, 3 regions, 2021-07-05 17:00:00 -> 2026-07-04 17:00:00


['demand_hour_utc',
 'respondent',
 'demand_mwh',
 'forecast_mwh',
 'demand_lag_1h',
 'demand_lag_24h',
 'demand_lag_168h',
 'demand_rolling_mean_24h',
 'demand_rolling_std_24h',
 'demand_rolling_mean_168h',
 'demand_rolling_std_168h',
 'hour_of_day',
 'day_of_week',
 'month',
 'is_weekend',
 'is_holiday']

In [19]:
out_path = PROJECT_ROOT / "data" / "raw" / "features_raw.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(out_path, index=False)

print(f"Wrote {out_path} ({len(df):,} rows)")

Wrote /Users/devashish/Desktop/Projects/gridcast/data/raw/features_raw.csv (131,402 rows)
